In [1]:
import os, csv, random
from collections import defaultdict, Counter

KG_DIR     = r'D:\Memoire\LLMKGER\kg'
OUT_DIR    = r'D:\Memoire\LLMKGER\kg_final'
os.makedirs(OUT_DIR, exist_ok=True)
random.seed(42)

# Parametres — AJUSTABLES
MIN_INTERACTIONS = 20   # user doit avoir suivi >= 20 cours
MIN_USERS        = 30   # cours doit avoir >= 30 users

print(f'MIN_INTERACTIONS = {MIN_INTERACTIONS}')
print(f'MIN_USERS        = {MIN_USERS}')


MIN_INTERACTIONS = 20
MIN_USERS        = 30


## Étape 1  Chargement des triplets


In [2]:
triples_by_rel = defaultdict(list)
with open(os.path.join(KG_DIR, 'triplets.tsv'), encoding='utf-8') as f:
    reader = csv.DictReader(f, delimiter='\t')
    for row in reader:
        triples_by_rel[row['relation']].append(
            (row['head'], row['relation'], row['tail'])
        )
print('Charges :')
for rel, t in triples_by_rel.items():
    print(f'  {rel:<20} {len(t):>10,}')


Charges :
  watches               1,347,769
  has_concept             624,683
  belongs_to              451,078
  taught_by                13,312
  offered_by                3,605
  follows              11,523,119
  prerequisite_of          84,968


## Étape 2 — Filtrage avec nouveaux seuils


In [3]:
# Compter interactions
user_course_count = Counter(h for h,r,t in triples_by_rel['follows'])
course_user_count = Counter(t for h,r,t in triples_by_rel['follows'])

# Filtrer users et cours
active_users   = {u for u,cnt in user_course_count.items() if cnt >= MIN_INTERACTIONS}
active_courses = {c for c,cnt in course_user_count.items() if cnt >= MIN_USERS}

print(f'Users  : {len(user_course_count):,} -> {len(active_users):,}')
print(f'Cours  : {len(course_user_count):,} -> {len(active_courses):,}')

# Filtrer toutes les relations
follows_f    = [(h,r,t) for h,r,t in triples_by_rel['follows']
                if h in active_users and t in active_courses]
watches_f    = [(h,r,t) for h,r,t in triples_by_rel['watches']
                if h in active_users]
active_ccids = {t for h,r,t in watches_f}

hasconcept_f = [(h,r,t) for h,r,t in triples_by_rel['has_concept']
                if h in active_ccids]
active_concepts = {t for h,r,t in hasconcept_f}

belongs_f    = [(h,r,t) for h,r,t in triples_by_rel['belongs_to']
                if h in active_concepts and t in active_courses]
prereq_f     = [(h,r,t) for h,r,t in triples_by_rel['prerequisite_of']
                if h in active_concepts and t in active_concepts]
taught_f     = [(h,r,t) for h,r,t in triples_by_rel['taught_by']
                if h in active_courses]
offered_f    = [(h,r,t) for h,r,t in triples_by_rel['offered_by']
                if h in active_courses]

sampled = {
    'watches'        : watches_f,
    'has_concept'    : hasconcept_f,
    'belongs_to'     : belongs_f,
    'taught_by'      : taught_f,
    'offered_by'     : offered_f,
    'follows'        : follows_f,
    'prerequisite_of': prereq_f,
}
total = sum(len(v) for v in sampled.values())
total_kg = total - len(follows_f)  # triplets KG hors interactions user-cours

print('\n=== KG Final ===')
print(f'  Users    : {len(active_users):,}')
print(f'  Courses  : {len(active_courses):,}')
print(f'  Videos   : {len(active_ccids):,}')
print(f'  Concepts : {len(active_concepts):,}')
print()
for rel, triples in sampled.items():
    print(f'  {rel:<20} {len(triples):>10,}')
print(f'  {"TOTAL":<20} {total:>10,}')
print(f'  {"TOTAL KG (hors follows)":<20} {total_kg:>10,}')
print()
if total_kg > 1_000_000:
    print('⚠️  Encore trop grand -> augmenter MIN_INTERACTIONS et MIN_USERS')
elif total_kg < 100_000:
    print('⚠️  Trop petit -> diminuer les seuils')
else:
    print('✅  Taille correcte pour entrainement KGE !')


Users  : 3,211,048 -> 106,735
Cours  : 3,781 -> 2,822

=== KG Final ===
  Users    : 106,735
  Courses  : 2,822
  Videos   : 17,959
  Concepts : 100,109

  watches                 112,038
  has_concept             251,334
  belongs_to              211,691
  taught_by                10,742
  offered_by                2,793
  follows               5,258,360
  prerequisite_of          59,751
  TOTAL                 5,906,709
  TOTAL KG (hors follows)    648,349

✅  Taille correcte pour entrainement KGE !


## Étape 3 — Split train/valid/test (Leave-One-Out)


In [4]:
user_interactions = defaultdict(list)
for h, r, t in follows_f:
    user_interactions[h].append(t)

train_inter, valid_inter, test_inter = [], [], []
for uid, course_list in user_interactions.items():
    random.shuffle(course_list)
    if len(course_list) >= 3:
        test_inter.append( (uid, 'follows', course_list[0]))
        valid_inter.append((uid, 'follows', course_list[1]))
        for c in course_list[2:]:
            train_inter.append((uid, 'follows', c))
    else:
        for c in course_list:
            train_inter.append((uid, 'follows', c))

print(f'Train : {len(train_inter):,}')
print(f'Valid : {len(valid_inter):,}')
print(f'Test  : {len(test_inter):,}')


Train : 5,044,890
Valid : 106,735
Test  : 106,735


## Étape 4 — Export format PyKEEN (strings) + format numérique (bonus)


In [5]:
# ── KG triplets (hors follows) + train follows ────────────────────────────
kg_triples = []
for rel, triples in sampled.items():
    if rel != 'follows':
        kg_triples.extend(triples)
kg_triples.extend(train_inter)

# ── Export format STRING (PyKEEN) ─────────────────────────────────────────
def write_tsv(path, data, header):
    with open(path, 'w', encoding='utf-8', newline='') as f:
        w = csv.writer(f, delimiter='\t')
        w.writerow(header)
        for row in data:
            w.writerow(row)

write_tsv(os.path.join(OUT_DIR, 'kg_train.tsv'),
          kg_triples, ['head','relation','tail'])
write_tsv(os.path.join(OUT_DIR, 'interactions_train.tsv'),
          train_inter, ['user_id','relation','course_id'])
write_tsv(os.path.join(OUT_DIR, 'interactions_valid.tsv'),
          valid_inter, ['user_id','relation','course_id'])
write_tsv(os.path.join(OUT_DIR, 'interactions_test.tsv'),
          test_inter,  ['user_id','relation','course_id'])
print('Format STRING (PyKEEN) exporte')

# ── entity2id et relation2id ──────────────────────────────────────────────
all_entities = set()
for h, r, t in kg_triples:
    all_entities.add(h)
    all_entities.add(t)
entity2id   = {e: i for i, e in enumerate(sorted(all_entities))}
relation2id = {r: i for i, r in enumerate(sorted(sampled.keys()))}

write_tsv(os.path.join(OUT_DIR, 'entity2id.tsv'),
          entity2id.items(), ['entity','id'])

# relations.tsv format correct (id -> name)
with open(os.path.join(OUT_DIR, 'relation2id.tsv'), 'w', encoding='utf-8') as f:
    for name, idx in relation2id.items():
        f.write(f'{idx}\t{name}\n')
print('entity2id et relation2id exportes')

# ── Format NUMERIQUE (OpenKE / LibKGE) ───────────────────────────────────
num_dir = os.path.join(OUT_DIR, 'numeric_format')
os.makedirs(num_dir, exist_ok=True)

def write_numeric(path, data):
    with open(path, 'w', encoding='utf-8', newline='') as f:
        w = csv.writer(f, delimiter='\t')
        for h, r, t in data:
            h_id = entity2id.get(h, -1)
            r_id = relation2id.get(r, -1)
            t_id = entity2id.get(t, -1)
            if h_id >= 0 and r_id >= 0 and t_id >= 0:
                w.writerow([h_id, r_id, t_id])

write_numeric(os.path.join(num_dir, 'train2id.txt'), kg_triples)
write_numeric(os.path.join(num_dir, 'valid2id.txt'), valid_inter)
write_numeric(os.path.join(num_dir, 'test2id.txt'),  test_inter)
print('Format NUMERIQUE (OpenKE) exporte')

# ── Recap final ───────────────────────────────────────────────────────────
print('\n=== Fichiers exportes ===')
for root, dirs, files in os.walk(OUT_DIR):
    level = root.replace(OUT_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for fname in sorted(files):
        fpath = os.path.join(root, fname)
        size  = round(os.path.getsize(fpath)/1024/1024, 2)
        print(f'{indent}  {fname:<40} {size} MB')


Format STRING (PyKEEN) exporte
entity2id et relation2id exportes
Format NUMERIQUE (OpenKE) exporte

=== Fichiers exportes ===
kg_final/
  entity2id.tsv                            6.69 MB
  interactions_test.tsv                    2.96 MB
  interactions_train.tsv                   139.8 MB
  interactions_valid.tsv                   2.96 MB
  kg_train.tsv                             180.17 MB
  relation2id.tsv                          0.0 MB
  numeric_format/
    test2id.txt                              1.63 MB
    train2id.txt                             86.27 MB
    valid2id.txt                             1.63 MB
